# Independent pretrained DiffJev adaptations

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import sys
from pathlib import Path

# 与 notebook00 使用同一个项目目录
ROOT = Path("/content/drive/MyDrive/DiffJev")
assert (ROOT / "AGENTS.md").is_file(), f"项目目录不存在：{ROOT}"

os.chdir(ROOT)
os.environ["DIFFJEV_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from diffjev.notebook_runtime import NotebookRuntime, install_environment

ENV_DIR = Path("/content/diffjev-env")
WORKER = ENV_DIR / "bin/python"

if not WORKER.is_file():
    # 新运行时：只重建依赖环境，不下载模型、不重新准备数据
    runtime = install_environment(ROOT, ENV_DIR)
else:
    runtime = NotebookRuntime(ROOT, WORKER)
    runtime.save_settings()

print("项目目录：", ROOT)
print("实验 Python：", runtime.python)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
项目目录： /content/drive/MyDrive/DiffJev
实验 Python： /content/diffjev-env/bin/python


In [3]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='02_train_diffjev_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='02_train_diffjev')
print('Experiment worker:', runtime.python)


{'python': '3.12.3', 'packages': {'torch': '2.9.1', 'transformers': '5.17.0', 'datasets': '4.4.1', 'huggingface_hub': '1.5.0', 'accelerate': '1.12.0', 'numpy': '2.2.6', 'jeva': None}, 'devices': [{'name': 'NVIDIA A100-SXM4-80GB', 'total_memory': 85094825984}]}
Pinned environment imports passed. No GPU is required for preparation.
Experiment worker: /content/diffjev-env/bin/python


These cells perform real supervised training. The saved source/augmentation trace, 3,125 updates, final-four-layer-plus-normalization policy, development checkpoint rule, and separate calibration split are shared. Set RUNS to the machines/runs allocated to this session. Three anchor seeds are planned; the size panel initially uses seed 0.

In [4]:
run_python(r"""
RUNS = cfg['backbones']
for backbone in RUNS:
    summary = w.train_run(backbone, seed=SEED, device=DEVICE)
    print(backbone, summary)
    gc.collect()
""")


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 3009.85it/s]


KeyboardInterrupt: 

Resume a specific interrupted adaptation with w.train_run(..., resume=".../step_500.pt"). CPU/CUDA/Python/NumPy RNG, corruption generator, augmentation generator, counters, and trace identity are restored.